In [74]:
import sys
!{sys.executable} -m pip install keras-tuner

You should consider upgrading via the 'c:\Users\aboyc\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [75]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [76]:
data = pd.read_csv('C:\\Users\\aboyc\\ANN dude\\huge_1M_titanic.csv')

In [77]:
data.shape

(1000000, 12)

In [78]:
data = data.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
data['Embarked'] = data['Embarked'].replace({'S':'Southampton', 'C':'Cherbourg', 'Q':'Queenstown'})
data.dropna(subset=['Embarked'], inplace=True)
data['Fare']=data['Fare'].astype('int8')

In [79]:
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,1,female,NaN,0,0,76,Cherbourg
1,0,3,male,29.0,0,0,10,Southampton
2,0,3,male,20.0,0,0,12,Cherbourg
3,0,3,male,27.0,0,0,13,Southampton
4,0,3,male,32.0,0,0,4,Cherbourg


In [80]:
data.shape

(997760, 8)

In [81]:
label = LabelEncoder()

In [82]:
data['Sex'] = label.fit_transform(data['Sex'])

In [83]:
onehot = OneHotEncoder(sparse_output=False)

In [84]:
Embarked = onehot.fit_transform(data[['Embarked']])

In [85]:
Embarked = pd.DataFrame(Embarked, columns=onehot.get_feature_names_out(['Embarked']))

In [86]:
scale = StandardScaler()

In [87]:
num_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']

In [88]:
data[num_cols] = scale.fit_transform(data[num_cols])

In [89]:
data.shape

(997760, 8)

In [90]:
data.isnull().sum()

Survived         0
Pclass           0
Sex              0
Age         198157
SibSp            0
Parch            0
Fare             0
Embarked         0
dtype: int64

In [91]:
data.isnull().sum()

Survived         0
Pclass           0
Sex              0
Age         198157
SibSp            0
Parch            0
Fare             0
Embarked         0
dtype: int64

In [92]:
X= data.drop(columns=['Survived'])
y=data['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   

In [93]:
model = Sequential([
    Dense(128, input_shape=(X_train.shape[1],), activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

c:\Users\aboyc\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [94]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [96]:
X_train = X_train.drop(columns=['Embarked'])
X_train = pd.concat([X_train, Embarked.iloc[X_train.index]], axis=1)

X_test = X_test.drop(columns=['Embarked'])
X_test = pd.concat([X_test, Embarked.loc[X_test.index]], axis=1)

model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

KeyError: "['Embarked'] not found in axis"

In [97]:
def build_model(hp):

    model=Sequential([Dense(64, input_shape=(X_train.shape[1],), activation='relu'),
                      Dense(32, activation='relu'),
                      Dense(1, activation='sigmoid')])

optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop','Adamax','Adadelta'])


model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

return model


NameError: name 'hp' is not defined

In [98]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, executions_per_trial=3, directory='my_dir', project_name='titanic_tuning')

Reloading Tuner from my_dir\titanic_tuning\tuner0.json


In [99]:
tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

In [100]:
tuner.get_best_hyperparameters(num_trials=1)[0].values

{}

In [101]:
model = tuner.get_best_models(num_trials=1)[0].summary()

TypeError: Tuner.get_best_models() got an unexpected keyword argument 'num_trials'

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

Epoch 1/10


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense" is incompatible with the layer: expected axis -1 of input shape to have value 7, but received input with shape (None, 6)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 6), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>